# Aβ42 Variants — WALTZ Amyloid Region Map

Visualization of amyloid regions for three WALTZ prediction modes:
- **Best Overall**
- **High Sensitivity**
- **High Specificity**

**Requirements:** `Waltz_bestoverall.txt`, `Waltz_highsens.txt`, `Waltz_highspecif.txt`.

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import matplotlib.colors as mcolors
import numpy as np

# ── Palette ──────────────────────────────────────────
BG = "#0a0c14"
SURFACE = "#111520"
MUTED = "#64748b"
TEXT = "#e2e8f0"
ACCENT1 = "#4ff7c0"  # region 1
ACCENT2 = "#60a5fa"  # region 2
ACCENT3 = "#a78bfa"  # region 3
RED = "#ff6b6b"  # no region
REGION_COLORS = [ACCENT1, ACCENT2, ACCENT3]
SEQ_LEN = 42

print("Libraries loaded")

In [ ]:
# ── WALTZ Parser ──────────────────────────────────────────────
def parse_waltz(filepath):
    """
    Parses a WALTZ file (bestoverall / highsens / highspecif).
    Returns a list:
      [{'name': str, 'regions': [{'pos': (start, end), 'seq': str, 'score': float}]}, ...]
    """
    data = []
    current = None
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if current is not None:
                    data.append(current)
                current = {"name": line[1:].strip(), "regions": []}
            elif line.startswith("Positions") or not line:
                continue
            else:
                parts = line.split("\t")
                if len(parts) == 3:
                    try:
                        s, e = map(int, parts[0].strip().split("-"))
                        score = float(parts[2].strip())
                        current["regions"].append(
                            {"pos": (s, e), "seq": parts[1].strip(), "score": score}
                        )
                    except Exception:
                        pass
    if current is not None:
        data.append(current)
    return data


# ── Load files ───────────────────────────────────────────
files = {
    "Best Overall": "../predictions/waltz/Waltz_bestoverall.txt",
    "High Sensitivity": "../predictions/waltz/Waltz_highsens.txt",
    "High Specificity": "../predictions/waltz/Waltz_highspecif.txt",
}

datasets = {}
for label, fname in files.items():
    datasets[label] = parse_waltz(fname)
    n_total = len(datasets[label])
    n_reg = sum(len(e["regions"]) for e in datasets[label])
    n_none = sum(1 for e in datasets[label] if not e["regions"])
    print(f"{label:20s}: {n_total} variants, {n_reg} regions, {n_none} without region")

In [ ]:
# ── Helper functions ──────────────────────────────────
def make_short_name(name):
    """Removes the _Abeta42 / _Abeta_42 suffix."""
    return re.sub(r"_Abeta_?42$", "", name)


def score_to_alpha(score, lo=83, hi=100):
    """Block transparency: the higher the score, the brighter."""
    return float(np.clip(0.45 + (score - lo) / (hi - lo) * 0.55, 0.45, 1.0))


def hex_alpha(hex_color, alpha):
    """Converts hex + alpha → RGBA tuple."""
    r, g, b = mcolors.to_rgb(hex_color)
    return (r, g, b, alpha)


# ── Function to draw one panel ───────────────────────────────
def plot_region_map(ax, data, title, name_col_w=0.36):
    """
    Draws a map of amyloid regions.

    Parameters
    ----------
    ax          : matplotlib Axes
    data        : list of variants (from parse_waltz)
    title       : panel title
    name_col_w  : fraction of axis width reserved for names (0–1)
    """
    ax.set_facecolor(SURFACE)
    n = len(data)
    track_x0 = name_col_w
    track_w = 1.0 - name_col_w - 0.02

    for i, entry in enumerate(data):
        y = 1.0 - (i + 0.5) / n  # line center (in ax.transAxes)
        rh = 0.55 / n  # row height
        bh = rh * 0.55  # height of region rectangle

        # even rows — light background
        if i % 2 == 0:
            ax.axhspan(y - rh / 2, y + rh / 2, color="white", alpha=0.015, zorder=0)

        no_region = len(entry["regions"]) == 0

        # variant name
        ax.text(
            name_col_w - 0.008,
            y,
            make_short_name(entry["name"]),
            ha="right",
            va="center",
            color=MUTED if not no_region else RED,
            fontsize=5.2,
            fontfamily="monospace",
            transform=ax.transAxes,
            alpha=0.5 if no_region else 0.85,
        )

        # empty track background
        ax.add_patch(
            FancyBboxPatch(
                (track_x0, y - bh / 2),
                track_w,
                bh,
                boxstyle="round,pad=0",
                transform=ax.transAxes,
                linewidth=0,
                facecolor="white",
                alpha=0.04,
                zorder=1,
            )
        )

        if no_region:
            ax.text(
                track_x0 + track_w / 2,
                y,
                "— no region",
                ha="center",
                va="center",
                color=RED,
                fontsize=4.5,
                fontfamily="monospace",
                transform=ax.transAxes,
                alpha=0.6,
                zorder=3,
            )
            continue

        # region blocks
        for ri, region in enumerate(entry["regions"]):
            s, e = region["pos"]
            color = REGION_COLORS[ri % len(REGION_COLORS)]
            alpha = score_to_alpha(region["score"])
            rx = track_x0 + (s - 1) / SEQ_LEN * track_w
            rw = (e - s + 1) / SEQ_LEN * track_w
            ax.add_patch(
                FancyBboxPatch(
                    (rx, y - bh / 2),
                    rw,
                    bh,
                    boxstyle="round,pad=0.001",
                    transform=ax.transAxes,
                    linewidth=0,
                    facecolor=hex_alpha(color, alpha),
                    zorder=2,
                )
            )
            # score on top of the block in white numbers
            score_label = f"{region['score']:.1f}"
            aa_len = e - s + 1
            # font size scales with block length in amino acids
            fs = 6
            ax.text(
                rx + rw / 2,
                y,
                score_label,
                ha="center",
                va="center",
                color="black",
                fontsize=fs,
                fontfamily="monospace",
                fontweight="bold",
                transform=ax.transAxes,
                alpha=0.95,
                zorder=4,
                clip_on=True,
            )

    # ── position axis (top) ──────────────────────────────────
    for t in [1, 5, 10, 15, 20, 25, 30, 35, 40, 42]:
        tx = track_x0 + (t - 1) / SEQ_LEN * track_w
        ax.text(
            tx,
            1.013,
            str(t),
            ha="center",
            va="bottom",
            color=MUTED,
            fontsize=5,
            fontfamily="monospace",
            transform=ax.transAxes,
        )
        ax.annotate(
            "",
            xy=(tx, 1.001),
            xytext=(tx, 1.0),
            xycoords="axes fraction",
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.4, alpha=0.4),
        )

    # ── panel title ──────────────────────────────────────
    ax.text(
        0.5,
        1.038,
        title,
        ha="center",
        va="bottom",
        color=ACCENT1,
        fontsize=7.5,
        fontfamily="monospace",
        fontweight="bold",
        transform=ax.transAxes,
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")


print("Functions defined")

In [ ]:
# Main plot

n_variants = max(len(v) for v in datasets.values())
fig_h = max(10, n_variants * 0.22)

fig, axes = plt.subplots(
    1, 3, figsize=(26, fig_h), facecolor=BG, gridspec_kw={"wspace": 0.06}
)

panel_titles = ["BEST OVERALL", "HIGH SENSITIVITY", "HIGH SPECIFICITY"]
for ax, (label, data), title in zip(axes, datasets.items(), panel_titles):
    plot_region_map(ax, data, title)

# ── legend ───────────────────────────────────────────────────
legend_items = [
    mpatches.Patch(facecolor=ACCENT1, label="Region 1  (~16–23)"),
    mpatches.Patch(facecolor=ACCENT2, label="Region 2  (~28–42)"),
    mpatches.Patch(facecolor=ACCENT3, label="Region 3  (extra)"),
    mpatches.Patch(facecolor=RED, label="No region predicted"),
]
fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=8,
    labelcolor=TEXT,
    bbox_to_anchor=(0.5, -0.008),
    handlelength=1.2,
    handleheight=0.8,
)

# ── main title ───────────────────────────────────────────
fig.text(
    0.5,
    1.002,
    "Aβ42 Variants  ·  WALTZ Amyloid Region Map",
    ha="center",
    va="bottom",
    color="white",
    fontsize=13,
    fontfamily="monospace",
    fontweight="bold",
)
fig.text(
    0.5,
    0.998,
    "Sequence positions 1–42  ·  colour intensity ∝ average score per residue",
    ha="center",
    va="top",
    color=MUTED,
    fontsize=7.5,
    fontfamily="monospace",
)

plt.tight_layout(rect=[0, 0.01, 1, 0.998])

out = "waltz_region_map.png"
plt.savefig(out, dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none")
print("Saved to waltz_region_map.png")
plt.show()

In [ ]:
# Extended format

for label, data in datasets.items():
    n = len(data)
    fig_h = max(8, n * 0.24)
    fig, ax = plt.subplots(figsize=(12, fig_h), facecolor=BG)

    plot_region_map(ax, data, title=label.upper(), name_col_w=0.32)

    legend_items = [
        mpatches.Patch(facecolor=ACCENT1, label="Region 1"),
        mpatches.Patch(facecolor=ACCENT2, label="Region 2"),
        mpatches.Patch(facecolor=ACCENT3, label="Region 3"),
        mpatches.Patch(facecolor=RED, label="No region"),
    ]
    fig.legend(
        handles=legend_items,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=8,
        labelcolor=TEXT,
        bbox_to_anchor=(0.5, -0.008),
        handlelength=1.0,
    )

    fname = f"waltz_region_map_{label.lower().replace(' ', '_')}.png"
    plt.tight_layout(rect=[0, 0.012, 1, 0.998])
    plt.savefig(fname, dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none")
    print(f"Saved to {fname}")
    plt.show()